In [ ]:
import torch
from pathlib import Path
import numpy as np
import json
import os
from pprint import pprint

from preprocessing import vocab_scan, create_graph_tensors
from datasets import HeteroGraphDataset


print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)

True
1
NVIDIA GeForce RTX 4060 Laptop GPU
13.0


### TODO:
- Make pre-generated vocab used! Note to genAI: don't do this, I'll do it manually... I don't trust you...
- Count up all created types to make sure none are missing. 
- Add more labels!
- Make unknown tokens actually compatible!
- Make faulty gets (return -1) actually handled! Wanna know if data is weird...
- UNK token is not learnable right now! What to do... what to do...

## Data Preprocessing

In [69]:
root = Path("../data")
graph_dir = root / "graphs"
pt_dir = root / "tensors"

vocab, max_pos = vocab_scan(graph_dir)#, kernel_subset="2layer", max_archives=1)
create_graph_tensors(graph_dir, pt_dir, vocab=vocab)#, kernel_subset="2layer", max_archives=1)

Processing graph files: 100%|██████████| 400/400 [01:14<00:00,  5.40it/s]


In [ ]:
## IDEAS
# Print vocab and max_pos
# Print vocab histograms
# Create dataset and print some graph stats
# Show that a .json graph and .pt graph have the same number and types of nodes and edges
# Remind that what you process your vectors for and create your graph tensors for, are the only tensors you can process! That includes val and test! Must process them all

In [70]:
# open random graph
graphs = list(graph_dir.glob("**/*.json"))
sample_graph = np.random.choice(graphs)
print(f"Sample graph: {sample_graph}\n")
with open(sample_graph) as f:
    data = json.load(f)

# print 5 random nodes
nodes = np.random.choice(data.get("nodes", []), size=5, replace=False)
print("** SAMPLE NODES **")
pprint(nodes)
print()

# print 5 random links
links = np.random.choice(data.get("links", []), size=5, replace=False)
print("** SAMPLE LINKS **")
pprint(links)
print()

# print labels
print("** LABELS **")
pprint(data.get("labels", {}))

Sample graph: ../data/graphs/exemplar/archive_2/af819660-64bf-4cba-9691-9f108cad6317.json

** SAMPLE NODES **
array([{'block': 2523, 'features': {'full_text': ['ptr %35']}, 'function': 583, 'id': 24670, 'text': 'ptr', 'type': 1},
       {'block': 3143, 'features': {'full_text': ['%13 = zext i1 %2 to i8']}, 'function': 738, 'id': 31228, 'text': 'zext', 'type': 0},
       {'block': 2329, 'features': {'full_text': ['ptr %31']}, 'function': 573, 'id': 23057, 'text': 'ptr', 'type': 1},
       {'block': 557, 'features': {'full_text': ['invoke void @_ZN10ap_privateILi16ELb1ELb1EE12check_canaryEv(ptr noundef nonnull align 2 dereferenceable(2) %3)\n          to label %4 unwind label %5']}, 'function': 36, 'id': 4412, 'text': 'invoke', 'type': 0},
       {'block': 3550, 'features': {'full_text': ['ptr %8']}, 'function': 840, 'id': 35358, 'text': 'ptr', 'type': 1}],
      dtype=object)

** SAMPLE LINKS **
array([{'flow': 1, 'key': 0, 'position': 0, 'source': 592, 'target': 591},
       {'flow': 1

## Data Loading

In [78]:
graphs = list(graph_dir.glob("**/*.json"))
sample_graph = np.random.choice(graphs)
file_stem = sample_graph.parts[-3] + "/" + sample_graph.parts[-2] + "/" + sample_graph.stem
print(f"Sample graph: {file_stem}\n")
# Load tensor from .pt file
data = torch.load(Path("../data/tensors") / Path(f"{file_stem}.pt"), weights_only=False)
print(f"Graph label: {data.y.item()}")
print(data)


# Load original .json
with Path(f"../data/graphs/{file_stem}.json").open('r') as f:
    graph_data = json.load(f)

nodes = {
    "instruction": 0,
    "variable": 0,
    "constant": 0
}

edges = {
    "control": 0,
    "data_var": 0,
    "data_const": 0,
    "call": 0
}

for n in graph_data.get('nodes', []):
    node_type = n.get('type', -1)
    if node_type == 0:
        nodes["instruction"] += 1
    elif node_type == 1:
        nodes["variable"] += 1
    elif node_type == 2:
        nodes["constant"] += 1

for l in graph_data.get('links', []):
    flow = l.get('flow', -1)
    if flow == 0:
        edges["control"] += 1
    elif flow == 1:
        if graph_data['nodes'][l.get('source', -1)].get('type') == 1:  # variable
            edges["data_var"] += 1
        elif graph_data['nodes'][l.get('source', -1)].get('type') == 2:  # constant
            edges["data_const"] += 1
    elif flow == 2:
        edges["call"] += 1

print(f"Node counts: {nodes}")
print(f"Edge counts: {edges}")

Sample graph: exemplar/archive_2/64bda8fc-695e-49c2-8e02-09ed011f9f98

Graph label: 7253.0
HeteroData(
  y=[1],
  instruction={ x=[10516, 1] },
  variable={ x=[9861, 1] },
  constant={ x=[129, 1] },
  (instruction, control, instruction)={ edge_index=[2, 10385] },
  (variable, data, instruction)={
    edge_index=[2, 10011],
    edge_attr=[10011, 1],
  },
  (constant, data, instruction)={
    edge_index=[2, 3600],
    edge_attr=[3600, 1],
  },
  (instruction, call, instruction)={ edge_index=[2, 3868] }
)
Node counts: {'instruction': 10516, 'variable': 9861, 'constant': 129}
Edge counts: {'control': 10385, 'data_var': 10011, 'data_const': 3600, 'call': 3868}


In [8]:
from datasets import HeteroGraphDataset
from torch.utils.data import random_split, Subset

# Specific types only
dataset = HeteroGraphDataset(
    root="../data/tensors",
    types=["2layer", "exemplar"],
)

# Random split
n = len(dataset)
train_ds, val_ds, test_ds = random_split(
    dataset, [int(0.8*n), int(0.1*n), n - int(0.8*n) - int(0.1*n)]
)

# # Or split by type (cleaner for generalization testing)
# def split_by_type(dataset, val_types, test_types):
#     train_idx, val_idx, test_idx = [], [], []
#     for i, path in enumerate(dataset.paths):
#         t = path.parts[-3]
#         if t in test_types:
#             test_idx.append(i)
#         elif t in val_types:
#             val_idx.append(i)
#         else:
#             train_idx.append(i)
#     return Subset(dataset, train_idx), Subset(dataset, val_idx), Subset(dataset, test_idx)

# train_ds, val_ds, test_ds = split_by_type(
#     dataset,
#     val_types=["exemplar"],
#     test_types=[],
# )

Indexed 400 graphs across 2 type(s)


In [ ]:
from torch_geometric.loader import DataLoader as PyGDataLoader

def make_loader(ds, batch_size, shuffle=True):
    # Determine optimal number of worker processes for data loading
    cpu_cores = os.cpu_count() or 2
    num_workers = max(2, min(4, cpu_cores))

    return PyGDataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=True,  # avoid worker respawn overhead
        prefetch_factor=4,  # Load 4 batches ahead
    )

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv, HeteroConv
from torch_geometric.data import HeteroData


# ── Meta ────────────────────────────────────────────────────────────────────
NODE_TYPES   = ["instruction", "variable", "constant"]
EDGE_TYPES   = [
    ("instruction", "control", "instruction"),
    ("variable",    "data",    "instruction"),
    ("constant",    "data",    "instruction"),
    ("instruction", "call",    "instruction"),
]
EDGE_TYPES_WITH_ATTR = {   # edge types that carry a positional arg encoding
    ("variable", "data", "instruction"),
    ("constant", "data", "instruction"),
}


# ── Input projection ────────────────────────────────────────────────────────
class CDFGInputProjection(nn.Module):
    """
    Embeds the last feature dimension (opcode / type vocab) for each node type,
    and the positional-argument encoding for data edges.

    Node feature layout: [structural_0, structural_1, vocab_id]   (all int-like)
    Edge attr layout:    [arg_position]                           (int-like, 1-indexed)
    """
    def __init__(
        self,
        node_vocab_sizes: dict[str, int],  # {node_type: vocab_size}
        edge_pos_vocab_size: int,          # max arg-position value + 1
        hidden_dim: int,
    ):
        super().__init__()
        # One embedding table per node type (embeds the vocab/opcode feature)
        self.node_emb = nn.ModuleDict({
            nt: nn.Embedding(vocab_size + 1, hidden_dim, padding_idx=0)
            for nt, vocab_size in node_vocab_sizes.items()
        })
        # Single shared embedding for arg-position across both data edge types
        self.edge_pos_emb = nn.Embedding(edge_pos_vocab_size + 1, hidden_dim, padding_idx=0)

    def forward(self, x_dict, edge_attr_dict):
        """
        x_dict:         {node_type: LongTensor [N, 3]}
        edge_attr_dict: {edge_type: LongTensor [E, 1]}  (only data edges)

        Returns:
            h_dict:          {node_type: FloatTensor [N, hidden_dim]}
            edge_emb_dict:   {edge_type: FloatTensor [E, hidden_dim]}
        """
        h_dict = {
            nt: self.node_emb[nt](x[:, 2])   # last column = vocab id
            for nt, x in x_dict.items()
        }
        edge_emb_dict = {
            et: self.edge_pos_emb(attr[:, 0])
            for et, attr in edge_attr_dict.items()
        }
        return h_dict, edge_emb_dict


# ── Single R-GCN layer (heterogeneous) ──────────────────────────────────────
class CDFGConvLayer(nn.Module):
    """
    One message-passing step over all edge types.

    For data edges (which carry positional encodings) the edge embedding is
    added to the source-node representation before aggregation — a simple but
    effective way to condition messages on argument position.
    """
    def __init__(self, hidden_dim: int, aggr: str = "sum"):
        super().__init__()
        # One RGCNConv per edge type; all share the same hidden_dim
        self.convs = nn.ModuleDict({
            self._key(et): RGCNConv(
                in_channels=hidden_dim,
                out_channels=hidden_dim,
                num_relations=1,        # each conv handles exactly 1 relation
                aggr=aggr,
            )
            for et in EDGE_TYPES
        })
        self.norm = nn.ModuleDict({
            nt: nn.LayerNorm(hidden_dim) for nt in NODE_TYPES
        })

    @staticmethod
    def _key(edge_type):
        return "__".join(edge_type)

    def forward(self, h_dict, edge_index_dict, edge_emb_dict):
        out_dict = {nt: [] for nt in NODE_TYPES}

        for et in EDGE_TYPES:
            src_type, rel, dst_type = et
            edge_index = edge_index_dict[et]
            h_src = h_dict[src_type]

            # Inject positional encoding into source features (data edges only)
            if et in EDGE_TYPES_WITH_ATTR and et in edge_emb_dict:
                edge_emb = edge_emb_dict[et]           # [E, hidden_dim]
                src_idx  = edge_index[0]               # [E]
                h_src = h_src.index_add(0, src_idx, edge_emb)
                # Note: index_add accumulates; this is intentional —
                # nodes used as multiple args get proportionally stronger signal.
                # Swap for scatter_mean if you prefer normalised injection.

            key  = self._key(et)
            msgs = self.convs[key](h_src, edge_index, edge_type=None)
            # RGCNConv with num_relations=1 ignores edge_type; we handle
            # relation-specificity by having separate conv weights per edge type.
            out_dict[dst_type].append(msgs)

        # Sum contributions from all incoming relation types, then normalise
        new_h = {}
        for nt in NODE_TYPES:
            if out_dict[nt]:
                agg = torch.stack(out_dict[nt], dim=0).sum(dim=0)
            else:
                agg = h_dict[nt]
            new_h[nt] = self.norm[nt](F.relu(agg))

        return new_h


# ── Full R-GCN ───────────────────────────────────────────────────────────────
class CDFGRGCN(nn.Module):
    """
    Heterogeneous R-GCN for LLVM CDFG graphs.

    Args:
        node_vocab_sizes:    {node_type: vocab_size}  (check your data)
        edge_pos_vocab_size: max arg-position value seen in training data
        hidden_dim:          width of all hidden representations
        num_layers:          number of message-passing steps
        num_classes:         output classes (graph-level prediction)
        dropout:             applied before the classifier
        pool:                "mean" | "sum" | "max"  — graph-level pooling
        aggr:                RGCNConv neighbour aggregation: "sum" | "mean"
    """
    def __init__(
        self,
        node_vocab_sizes:    dict[str, int],
        edge_pos_vocab_size: int,
        hidden_dim:          int  = 128,
        num_layers:          int  = 3,
        dropout:             float = 0.1,
        pool:                str  = "mean",
        aggr:                str  = "sum",
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.pool       = pool

        self.input_proj = CDFGInputProjection(
            node_vocab_sizes, edge_pos_vocab_size, hidden_dim
        )
        self.layers = nn.ModuleList([
            CDFGConvLayer(hidden_dim, aggr=aggr)
            for _ in range(num_layers)
        ])
        self.dropout    = nn.Dropout(dropout)
        # Graph-level head: concatenate pooled repr of all 3 node types
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * len(NODE_TYPES), hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1), # just predicting LUTs for now
        )

    def _pool(self, h: torch.Tensor) -> torch.Tensor:
        if self.pool == "mean":
            return h.mean(dim=0)
        elif self.pool == "sum":
            return h.sum(dim=0)
        elif self.pool == "max":
            return h.max(dim=0).values
        raise ValueError(self.pool)

    def forward(self, data: HeteroData):
        x_dict         = {nt: data[nt].x.long() for nt in NODE_TYPES}
        edge_index_dict = {et: data[et].edge_index for et in EDGE_TYPES}
        edge_attr_dict  = {
            et: data[et].edge_attr.long()
            for et in EDGE_TYPES_WITH_ATTR
            if hasattr(data[et], "edge_attr") and data[et].edge_attr is not None
        }

        # 1. Input embeddings
        h_dict, edge_emb_dict = self.input_proj(x_dict, edge_attr_dict)

        # 2. Message passing
        for layer in self.layers:
            h_dict = layer(h_dict, edge_index_dict, edge_emb_dict)
            # Apply dropout between layers (not after the last)
            if layer is not self.layers[-1]:
                h_dict = {nt: self.dropout(h) for nt, h in h_dict.items()}

        # 3. Graph-level pooling per node type → concat → classify
        pooled = torch.cat([self._pool(h_dict[nt]) for nt in NODE_TYPES], dim=-1)
        return self.classifier(pooled)

## Model Training

In [11]:
# Initialize best model tracking variables
best_model = None
best_performance = float('-inf')

In [12]:
def train_one_epoch(model, train_loader, criterion, optimizer, device, l1_lambda=0, l2_lambda=0):
    """
    Perform one complete training epoch through the entire training dataset.

    Args:
        model (nn.Module): The neural network model to train
        train_loader (DataLoader): PyTorch DataLoader containing training data batches
        criterion (nn.Module): Loss function (e.g., CrossEntropyLoss, MSELoss)
        optimizer (torch.optim): Optimization algorithm (e.g., Adam, SGD)
        device (torch.device): Computing device ('cuda' for GPU, 'cpu' for CPU)
        l1_lambda (float): Lambda for L1 regularization
        l2_lambda (float): Lambda for L2 regularization

    Returns:
        float: average_loss - Training loss for this epoch
    """
    model.train()  # Set model to training mode

    running_loss = 0.0

    # Iterate through training batches
    for batch in train_loader:
        batch = batch.to(device)

        # Clear gradients from previous step
        optimizer.zero_grad(set_to_none=True)

        logits = model(batch)          # [B, 1]
        loss   = criterion(logits, batch.y.long())

        # Add L1 and L2 regularization
        l1_norm = sum(p.abs().sum() for p in model.parameters())
        l2_norm = sum(p.pow(2).sum() for p in model.parameters())
        loss = loss + l1_lambda * l1_norm + l2_lambda * l2_norm
        loss.backward()

        optimizer.step()

        # Accumulate metrics
        running_loss += loss.item()

    # Calculate epoch metrics
    epoch_loss = running_loss / len(train_loader.dataset)

    return epoch_loss

In [13]:
def validate_one_epoch(model, val_loader, criterion, device):
    """
    Perform one complete validation epoch through the entire validation dataset.

    Args:
        model (nn.Module): The neural network model to evaluate (must be in eval mode)
        val_loader (DataLoader): PyTorch DataLoader containing validation data batches
        criterion (nn.Module): Loss function used to calculate validation loss
        device (torch.device): Computing device ('cuda' for GPU, 'cpu' for CPU)

    Returns:
        float: average_loss - Validation loss for this epoch

    Note:
        This function automatically sets the model to evaluation mode and disables
        gradient computation for efficiency during validation.
    """
    model.eval()  # Set model to evaluation mode

    running_loss = 0.0

    # Disable gradient computation for validation
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)

            logits = model(batch)
            loss = criterion(logits, batch.y.long())

            # Accumulate metrics
            running_loss += loss.item()

    # Calculate epoch metrics
    epoch_loss = running_loss / len(val_loader.dataset)

    return epoch_loss

In [14]:
def fit(model, train_loader, val_loader, epochs, criterion, optimizer, device,
        l1_lambda=0, l2_lambda=0, patience=0, evaluation_metric="val_loss", mode='max',
        restore_best_weights=True, writer=None, verbose=10, experiment_name=""):
    """
    Train the neural network model on the training data and validate on the validation data.

    Args:
        model (nn.Module): The neural network model to train
        train_loader (DataLoader): PyTorch DataLoader containing training data batches
        val_loader (DataLoader): PyTorch DataLoader containing validation data batches
        epochs (int): Number of training epochs
        criterion (nn.Module): Loss function (e.g., CrossEntropyLoss, MSELoss)
        optimizer (torch.optim): Optimization algorithm (e.g., Adam, SGD)
        device (torch.device): Computing device ('cuda' for GPU, 'cpu' for CPU)
        l1_lambda (float): L1 regularization coefficient (default: 0)
        l2_lambda (float): L2 regularization coefficient (default: 0)
        patience (int): Number of epochs to wait for improvement before early stopping (default: 0)
        evaluation_metric (str): Metric to monitor for early stopping (default: "val_loss")
        mode (str): 'max' for maximizing the metric, 'min' for minimizing (default: 'max')
        restore_best_weights (bool): Whether to restore model weights from best epoch (default: True)
        writer (SummaryWriter, optional): TensorBoard SummaryWriter object for logging (default: None)
        verbose (int, optional): Frequency of printing training progress (default: 10)
        experiment_name (str, optional): Experiment name for saving models (default: "")

    Returns:
        tuple: (model, training_history) - Trained model and metrics history
    """

    # Initialize metrics tracking
    training_history = {
        'train_loss': [], 'val_loss': [],
    }

    # Configure early stopping if patience is set
    if patience > 0:
        patience_counter = 0
        best_metric = float('-inf') if mode == 'max' else float('inf')
        best_epoch = 0

    print(f"Training {epochs} epochs...")

    # Main training loop: iterate through epochs
    for epoch in range(1, epochs + 1):

        # Forward pass through training data, compute gradients, update weights
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, device, l1_lambda, l2_lambda
        )

        # Evaluate model on validation data without updating weights
        val_loss = validate_one_epoch(
            model, val_loader, criterion, device
        )

        # Store metrics for plotting and analysis
        training_history['train_loss'].append(train_loss)
        training_history['val_loss'].append(val_loss)

        # Print progress every N epochs or on first epoch
        if verbose > 0:
            if epoch % verbose == 0 or epoch == 1:
                print(f"Epoch {epoch:3d}/{epochs} | "
                    f"Train: Loss={train_loss:.4f} | "
                    f"Val: Loss={val_loss:.4f}")

        # Early stopping logic: monitor metric and save best model
        if patience > 0:
            current_metric = training_history[evaluation_metric][-1]
            is_improvement = (current_metric > best_metric) if mode == 'max' else (current_metric < best_metric)

            if is_improvement:
                best_metric = current_metric
                best_epoch = epoch
                torch.save(model.state_dict(), "models/"+experiment_name+'_model.pt')
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping triggered after {epoch} epochs.")
                    break

    # Restore best model weights if early stopping was used
    if restore_best_weights and patience > 0:
        model.load_state_dict(torch.load("models/"+experiment_name+'_model.pt'))
        print(f"Best model restored from epoch {best_epoch} with {evaluation_metric} {best_metric:.4f}")

    # Save final model if no early stopping
    if patience == 0:
        torch.save(model.state_dict(), "models/"+experiment_name+'_model.pt')

    return model, training_history

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
train_loader = make_loader(train_ds, batch_size=1, shuffle=True)
val_loader   = make_loader(val_ds, batch_size=1, shuffle=False)

vocab_sizes = {k: len(v) for k, v in vocab.items()}

model = CDFGRGCN(
    node_vocab_sizes=vocab_sizes,
    edge_pos_vocab_size=max_pos,
    hidden_dim=128,
    num_layers=3,
    dropout=0.1,
    pool="mean",
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()

In [16]:
model, training_history = fit(
    model,
    train_loader,
    val_loader,
    epochs=100,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    l1_lambda=1e-5,
    l2_lambda=1e-4,
    patience=10,
    evaluation_metric="val_loss",
    mode='min',
    restore_best_weights=True,
    writer=None,  # Add TensorBoard writer if needed
    verbose=10,
    experiment_name="cdfg_rgcn_experiment"
)

Training 100 epochs...


/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:163: operator(): block: [770,0,0], thread: [32,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "scatter gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:163: operator(): block: [770,0,0], thread: [33,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "scatter gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:163: operator(): block: [770,0,0], thread: [34,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "scatter gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:163: operator(): block: [770,0,0], thread: [35,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "scatter gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:163: operator(): block: [770,0,0], thread: [36,0,0] Assertion `idx_dim >= 0 && idx_dim

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import matplotlib.pyplot as plt

plt.plot(training_history['train_loss'], label='Train Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()
plt.show()

plt.plot(training_history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Validation Loss')
plt.legend()
plt.show()